## Compare RC15 with various minimal sequence length

In [34]:
import os
import re
import pandas as pd
from matplotlib import pyplot as plt

models = ['sasrec', 'caser', 'gru', 'nextitnet']
results_dir = "/home/marek/Kinit/my_smorl/Plots/Steps_data"
os.makedirs(results_dir, exist_ok=True)

In [46]:
patterns = {
    # Matches: cumulative reward @ 5: 8585.000000
    'cumulative_reward': re.compile(r'.*cumulative reward @ (\d+): ([\d.]+)$'),

    # Matches: clicks hr ndcg @ 10 : 0.398485, 0.241570
    'clicks_hr_ndcg': re.compile(r'.*clicks hr ndcg @ (\d+) ?: ([\d.]+), ([\d.]+)$'),

    # Matches: purchase hr and ndcg @10 : 0.535500, 0.337177
    'purchase_hr_ndcg': re.compile(r'.*purchase hr and ndcg @(\d+) ?: ([\d.]+), ([\d.]+)$'),

    # Matches: total diversity reward: 48740.039062
    'total_diversity': re.compile(r'.*total diversity reward: ([\d.]+)$'),

    # Matches: total novelty reward: 19004.000000
    'total_novelty': re.compile(r'.*total novelty reward: ([\d.]+)$'),

    # Matches: coverage of top 5 predictions: 0.379410
    'coverage': re.compile(r'.*coverage of top (\d+) predictions: ([\d.]+)$'),

    # Matches: coverage on novel items of top 10 predictions: 0.385095
    'novel_coverage': re.compile(r'.*coverage on novel items of top (\d+) predictions: ([\d.]+)$'),

    # Matches: average number of repetitions in top 20: 53.379500
    'avg_repetitions': re.compile(r'.*average number of repetitions in top (\d+): ([\d.]+)$'),
    
    'steps': re.compile(r'.*Step: (\d+)\.+\s+Loss: ([\d.]+)$')
}

In [48]:
def process_results(data_path, skip):
    eval_step = {'rc15_results': 5000, 'retail_rocket_results': 10000}
    
    loss = {
        'steps': [], 'loss': []
    }
    metrics = {
        'steps': [],
        'hr_val_5': [], 'hr_val_10': [], 'hr_val_20': [], 
        'ndcg_val_5': [], 'ndcg_val_10': [], 'ndcg_val_20': [],
        'cov_val_1': [], 'cov_val_5': [], 'cov_val_10': [], 'cov_val_20': [],        
        'nov_val_1': [], 'nov_val_5': [], 'nov_val_10': [], 'nov_val_20': [],
        'rep_val_5': [], 'rep_val_10': [], 'rep_val_20': [] 
    }
    with open(data_path, "r") as f:
        lines = f.readlines()
        skips = -1
        for line in lines:
            match = re.search(r".*START EVALUATION", line)
            if match:
                skips += 1
            if skips == 3:
                    skips = 0
            if m := patterns['steps'].match(line):
                step, _loss = int(m.group(1)), float(m.group(2))
                loss['steps'].append(step)
                loss['loss'].append(_loss)                
            if str(skips) == skip:
                if m := patterns['clicks_hr_ndcg'].match(line):
                    k, hr, ndcg = int(m.group(1)), float(m.group(2)), float(m.group(3))
                    metrics[f'hr_val_{k}'].append(hr)
                    metrics[f'ndcg_val_{k}'].append(ndcg)
                elif m := patterns['coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'cov_val_{k}'].append(val)
                elif m := patterns['novel_coverage'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'nov_val_{k}'].append(val)
                elif m := patterns['avg_repetitions'].match(line):
                    k, val = int(m.group(1)), float(m.group(2))
                    metrics[f'rep_val_{k}'].append(val)
        metrics['steps'] = [eval_step['rc15_results'] * (i + 1) for i in range(len(metrics['hr_val_5']))]  
    return loss, metrics

basepath = "/home/marek/Kinit/Paparella/SMORL/div4rec/rc15_results"
for model in models:
    for data_skip in ['0', '1', '2']:
        for eval_skip in ['0', '1', '2']:
            if model == 'gru':
                file_path = f'{basepath}/{model}_test_skip={data_skip}/{model}.txt'
            else:
                file_path = f'{basepath}/{model}_test_0_skip={data_skip}/{model}.txt'
            results_path = f'{results_dir}/{model}'
            os.makedirs(results_path, exist_ok=True)
            results = f"{results_path}/train_{data_skip}_test_{eval_skip}"
            losses, metrics = process_results(file_path, eval_skip)   
            df_loss = pd.DataFrame(losses)
            df_metrics = pd.DataFrame(metrics)
            df_loss.to_pickle(f'{results}_loss')
            df_loss.to_csv(f'{results}_loss.csv')
            df_metrics.to_pickle(f'{results}_metrics')
            df_metrics.to_csv(f'{results}_metrics.csv')


In [52]:
def plot_helper(ax, df0, df1, df2, df3, df4, df5, df6, df7, df8, PLOT, window=1):
    ax.set_ylabel(PLOT, color='black')
    ax.plot(df0['steps'], df0[PLOT].rolling(window).mean(), color='black', linewidth=1, label='train-0-eval-0')
    ax.plot(df1['steps'], df1[PLOT].rolling(window).mean(), color='black', linestyle="--", linewidth=2, label='train-0-eval-1')
    ax.plot(df2['steps'], df2[PLOT].rolling(window).mean(), color='black', linestyle=":", linewidth=2, label='train-0-eval-2')
    ax.plot(df3['steps'], df3[PLOT].rolling(window).mean(), color='green', linewidth=1, label='train-1-eval-0')
    ax.plot(df4['steps'], df4[PLOT].rolling(window).mean(), color='green', linestyle="--", linewidth=2, label='train-1-eval-1')
    ax.plot(df5['steps'], df5[PLOT].rolling(window).mean(), color='green', linestyle=":", linewidth=2, label='train-1-eval-2')
    ax.plot(df6['steps'], df6[PLOT].rolling(window).mean(), color='red', linewidth=1, label='train-2-eval-0')
    ax.plot(df7['steps'], df7[PLOT].rolling(window).mean(), color='red', linestyle="--", linewidth=2, label='train-2-eval-1')
    ax.plot(df8['steps'], df8[PLOT].rolling(window).mean(), color='red', linestyle=":", linewidth=2, label='train-2-eval-2')
    
def plot_metrics(basepath, model, window=1):  
    df0 = pd.read_pickle(f"{basepath}/{model}/train_0_test_0_metrics")
    df1 = pd.read_pickle(f"{basepath}/{model}/train_0_test_1_metrics")
    df2 = pd.read_pickle(f"{basepath}/{model}/train_0_test_2_metrics")
    df3 = pd.read_pickle(f"{basepath}/{model}/train_1_test_0_metrics")
    df4 = pd.read_pickle(f"{basepath}/{model}/train_1_test_1_metrics")
    df5 = pd.read_pickle(f"{basepath}/{model}/train_1_test_2_metrics")
    df6 = pd.read_pickle(f"{basepath}/{model}/train_2_test_0_metrics")
    df7 = pd.read_pickle(f"{basepath}/{model}/train_2_test_1_metrics")
    df8 = pd.read_pickle(f"{basepath}/{model}/train_2_test_2_metrics")
    
    #~~~~~~ Plot cov + nov values ~~~~~~
    fig, axs = plt.subplots(3, 2, figsize=(14, 14))
    fig.suptitle(f'Datasets and evals variations: {model}', fontsize=12, y=0.94)
     
    PLOT = f"cov_val_10"
    ax1 = axs[0, 0]
    ax1.set_title("COV 10", fontsize=10, pad=10)
    ax1.set_ylabel(PLOT, color='black')
    plot_helper(ax1, df0, df1, df2, df3, df4, df5, df6, df7, df8, PLOT)
    
    PLOT = f"nov_val_10"
    ax2 = axs[0, 1]
    ax2.set_title("NOV 10", fontsize=10, pad=10)
    plot_helper(ax2, df0, df1, df2, df3, df4, df5, df6, df7, df8, PLOT)
   
    PLOT = f"hr_val_10"
    ax3 = axs[1, 0]
    ax3.set_title("HR 10", fontsize=10, pad=10)
    plot_helper(ax3, df0, df1, df2, df3, df4, df5, df6, df7, df8, PLOT)
    
    PLOT = f"ndcg_val_10"
    ax4 = axs[1, 1]
    ax4.set_title("NDCG 10", fontsize=10, pad=10)
    plot_helper(ax4, df0, df1, df2, df3, df4, df5, df6, df7, df8, PLOT)
    
    PLOT = f"rep_val_5"
    ax5 = axs[2, 0]
    ax5.set_title("REP 5", fontsize=10, pad=10)
    plot_helper(ax5, df0, df1, df2, df3, df4, df5, df6, df7, df8, PLOT)
    
    # plot Loss
    dfl0 = pd.read_pickle(f"{basepath}/{model}/train_0_test_0_loss")
    dfl1 = pd.read_pickle(f"{basepath}/{model}/train_0_test_1_loss")
    dfl2 = pd.read_pickle(f"{basepath}/{model}/train_0_test_2_loss")
    dfl3 = pd.read_pickle(f"{basepath}/{model}/train_1_test_0_loss")
    dfl4 = pd.read_pickle(f"{basepath}/{model}/train_1_test_1_loss")
    dfl5 = pd.read_pickle(f"{basepath}/{model}/train_1_test_2_loss")
    dfl6 = pd.read_pickle(f"{basepath}/{model}/train_2_test_0_loss")
    dfl7 = pd.read_pickle(f"{basepath}/{model}/train_2_test_1_loss")
    dfl8 = pd.read_pickle(f"{basepath}/{model}/train_2_test_2_loss")
    
    PLOT = f"loss"
    ax6 = axs[2, 1]
    ax6.set_ylim(3, 10)
    ax6.set_title("Loss", fontsize=10, pad=10)
    plot_helper(ax6, dfl0, dfl1, dfl2, dfl3, dfl4, dfl5, dfl6, dfl7, dfl8, PLOT, window=10) 
    
    for i, ax in enumerate(axs.flat):
        ax.yaxis.grid(True, color='lightgray', linewidth=0.5)
        ax.tick_params(axis='y', labelsize=8)
    
    handles, labels = ax1.get_legend_handles_labels()
    seen = set()
    unique = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
    fig.legend(*zip(*unique), loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), fontsize='medium')
    fig.savefig(f"{basepath}/EVAL_STEP-{model}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)

filepath = f"{results_dir}"
for model in models:
    plot_metrics(filepath, model)